# Micrograd Practice

Goal:
Understand how neural networks learn by building the machinery ourselves.

In [1]:
class Value:
    def __init__(self, data):
        self.data = data

In [2]:
a = Value(3)
b = Value(2)

print(a.data)
print(b.data)

3
2


In [3]:
a = Value(3)
b = Value(2)

In [4]:
c = a + b

TypeError: unsupported operand type(s) for +: 'Value' and 'Value'

In [5]:
class Value:
    def __init__(self, data):
        self.data = data

    def __add__(self, other):
        return Value(self.data + other.data)

In [7]:
a = Value(3)
b = Value(2)

In [10]:
(a + b).data

5

In [ ]:
class Value:
    def __init__(self, data, children: set = None):
        self.data = data
        self._children = children

    def __add__(self, other):
        return Value(self.data + other.data, {self, other})

In [30]:
a = Value(3)
b = Value(2)
c = Value(5, (a, b))

In [31]:
c._children

(<__main__.Value at 0x10a38c2d0>, <__main__.Value at 0x10a3aa050>)

In [32]:
a = Value(3)
b = Value(2)
c = a + b

print(c.data)
print(c._children)

5
{<__main__.Value object at 0x10a3ab690>, <__main__.Value object at 0x10a331bd0>}


# Operation Remember

In [ ]:
class Value:
    def __init__(self, data, children: set = None, op: str = None):
        self.data = data
        self._children = children
        self.op = op
        self.grad = 0

    def __add__(self, other):
        return Value(self.data + other.data, {self, other}, "+")

In [33]:
a = Value(3)
b = Value(2)
c = a + b

c.grad = 1

# Backward

In [35]:
class Value:
    def __init__(self, data, children: set = None, op: str = None, backward_fn = None):
        self.data = data
        self._children = children
        self.op = op
        self.grad = 0
        self._backward = backward_fn

    def __add__(self, other):
        out = Value(self.data + other.data, {self, other}, "+")
        def _backward():
            self.grad += out.grad * 1
            other.grad += out.grad * 1
        out._backward = _backward
        return out

    def backward(self):
        self.grad = 1
        self._backward()

In [36]:
a = Value(3)
b = Value(2)
c = a + b

c.backward()

print(a.grad)
print(b.grad)
print(c.grad)

1
1
1


# topological ordering

In [11]:
class Value:
    def __init__(self, data, children: set = set(), op: str = None, backward_fn = lambda: None):
        self.data = data
        self._children = children
        self.op = op
        self.grad = 0
        self._backward = backward_fn

    def __add__(self, other):
        out = Value(self.data + other.data, {self, other}, "+")
        def _backward():
            self.grad += out.grad * 1
            other.grad += out.grad * 1
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        # print(topo)
        self.grad = 1
        for item in reversed(topo):
            item._backward()

In [12]:
a = Value(3)
b = Value(2)

c = a + b
d = c + b

d.backward()

# Multiplication

In [17]:
class Value:
    def __init__(self, data, children: set = set(), op: str = None, backward_fn = lambda: None):
        self.data = data
        self._children = children
        self.op = op
        self.grad = 0
        self._backward = backward_fn

    def __add__(self, other):
        out = Value(self.data + other.data, {self, other}, "+")
        def _backward():
            self.grad += out.grad * 1
            other.grad += out.grad * 1
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        # print(topo)
        self.grad = 1
        for item in reversed(topo):
            item._backward()

    def __mul__(self, other):
        out = Value(self.data * other.data, {self, other}, "*")
        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        return out

In [18]:
a = Value(3)
b = Value(2)
c = a * b
c.backward()

print(a.grad)
print(b.grad)
print(c.grad)

2
3
1


In [20]:
a = Value(2)
b = Value(3)
c = a * b
d = c + b
d.backward()
print(a.grad)
print(b.grad)
print(c.grad)
print(d.grad)

3
3
1
1


In [21]:
a = Value(3)
c = a + 2

AttributeError: 'int' object has no attribute 'data'

# Reverse addition

In [27]:
class Value:
    def __init__(self, data, children: set = set(), op: str = None, backward_fn = lambda: None):
        self.data = data
        self._children = children
        self.op = op
        self.grad = 0
        self._backward = backward_fn

    def __add__(self, other):
        if type(other) is not Value:
            other = Value(other)
        out = Value(self.data + other.data, {self, other}, "+")
        def _backward():
            self.grad += out.grad * 1
            other.grad += out.grad * 1
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        # print(topo)
        self.grad = 1
        for item in reversed(topo):
            item._backward()

    def __mul__(self, other):
        out = Value(self.data * other.data, {self, other}, "*")
        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        return out

    def __radd__(self, other):
        return self.__add__(other)

In [28]:
a = Value(3)
b = 2 + a
c = b * a
c.backward()
print(a.grad)
print(b.grad)
print(c.grad)
print(c.data)

8
3
1
15


# Subtraction and division

In [ ]:
class Value:
    def __init__(self, data, children: set = set(), op: str = None, backward_fn = lambda: None):
        self.data = data
        self._children = children
        self.op = op
        self.grad = 0
        self._backward = backward_fn

    def __add__(self, other):
        if type(other) is not Value:
            other = Value(other)
        out = Value(self.data + other.data, {self, other}, "+")
        def _backward():
            self.grad += out.grad * 1
            other.grad += out.grad * 1
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        # print(topo)
        self.grad = 1
        for item in reversed(topo):
            item._backward()

    def __mul__(self, other):
        if type(other) is not Value:
            other = Value(other)
        out = Value(self.data * other.data, {self, other}, "*")
        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        return out

    def __radd__(self, other):
        return self.__add__(other)

    def __sub__(self, other):
        if type(other) is not Value:
            other = Value(other)
        out = Value(self.data - other.data, {self, other}, "-")
        def _backward():
            self.grad += out.grad * 1
            other.grad -= out.grad * 1
        out._backward = _backward
        return out

    def __rsub__(self, other):
        if type(other) is not Value:
            other = Value(other)
        return other.__sub__(self)

    def __rmul__(self, other):
        return self.__mul__(other)

    def __truediv__(self, other):
        if type(other) is not Value:
            other = Value(other)
        out = Value(self.data / other.data, {self, other}, "/")
        def _backward():
            self.grad += out.grad / other.data
            other.grad += (-1) *out.grad * self.data / other.data ** 2
        out._backward = _backward
        return out

    def __rtruediv__(self, other):
        if type(other) is not Value:
            other = Value(other)
        return other.__truediv__(self)

In [33]:
a = Value(5)
b = Value(2)
c = a - b
c.backward()

In [34]:
print(c.data)
print(c.grad)
print(a.grad)
print(b.grad)

3
1
1
-1


In [35]:
a = Value(6)
b = Value(2)
c = a / b
c.backward()

In [36]:
print(c.data)
print(c.grad)
print(a.grad)
print(b.grad)

3.0
1
0.5
-1.5


# Power

In [38]:
import math

class Value:
    def __init__(self, data, children: set = set(), op: str = None, backward_fn = lambda: None):
        self.data = data
        self._children = children
        self.op = op
        self.grad = 0
        self._backward = backward_fn

    def __add__(self, other):
        if type(other) is not Value:
            other = Value(other)
        out = Value(self.data + other.data, {self, other}, "+")
        def _backward():
            self.grad += out.grad * 1
            other.grad += out.grad * 1
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        # print(topo)
        self.grad = 1
        for item in reversed(topo):
            item._backward()

    def __mul__(self, other):
        if type(other) is not Value:
            other = Value(other)
        out = Value(self.data * other.data, {self, other}, "*")
        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        return out

    def __radd__(self, other):
        return self.__add__(other)

    def __sub__(self, other):
        if type(other) is not Value:
            other = Value(other)
        out = Value(self.data - other.data, {self, other}, "-")
        def _backward():
            self.grad += out.grad * 1
            other.grad -= out.grad * 1
        out._backward = _backward
        return out

    def __rsub__(self, other):
        if type(other) is not Value:
            other = Value(other)
        return other.__sub__(self)

    def __rmul__(self, other):
        return self.__mul__(other)

    def __truediv__(self, other):
        if type(other) is not Value:
            other = Value(other)
        out = Value(self.data / other.data, {self, other}, "/")
        def _backward():
            self.grad += out.grad / other.data
            other.grad += (-1) *out.grad * self.data / other.data ** 2
        out._backward = _backward
        return out

    def __rtruediv__(self, other):
        if type(other) is not Value:
            other = Value(other)
        return other.__truediv__(self)

    def __pow__(self, other):
        out = Value(self.data ** other, {self}, "**")
        def _backward():
            self.grad += out.grad *  other * self.data ** (other - 1)
        out._backward = _backward
        return out

In [39]:
a = Value(2)
b = Value(3)

c = a * b
d = c + a
e = d ** 2

e.backward()

In [40]:
print(c.data)
print(d.data)
print(e.data)

print(e.grad)
print(d.grad)
print(c.grad)
print(a.grad)
print(b.grad)

6
8
64
1
16
16
64
32


# Tanh

In [41]:
import math

class Value:
    def __init__(self, data, children: set = set(), op: str = None, backward_fn = lambda: None):
        self.data = data
        self._children = children
        self.op = op
        self.grad = 0
        self._backward = backward_fn

    def __add__(self, other):
        if type(other) is not Value:
            other = Value(other)
        out = Value(self.data + other.data, {self, other}, "+")
        def _backward():
            self.grad += out.grad * 1
            other.grad += out.grad * 1
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        # print(topo)
        self.grad = 1
        for item in reversed(topo):
            item._backward()

    def __mul__(self, other):
        if type(other) is not Value:
            other = Value(other)
        out = Value(self.data * other.data, {self, other}, "*")
        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        return out

    def __radd__(self, other):
        return self.__add__(other)

    def __sub__(self, other):
        if type(other) is not Value:
            other = Value(other)
        out = Value(self.data - other.data, {self, other}, "-")
        def _backward():
            self.grad += out.grad * 1
            other.grad -= out.grad * 1
        out._backward = _backward
        return out

    def __rsub__(self, other):
        if type(other) is not Value:
            other = Value(other)
        return other.__sub__(self)

    def __rmul__(self, other):
        return self.__mul__(other)

    def __truediv__(self, other):
        if type(other) is not Value:
            other = Value(other)
        out = Value(self.data / other.data, {self, other}, "/")
        def _backward():
            self.grad += out.grad / other.data
            other.grad += (-1) *out.grad * self.data / other.data ** 2
        out._backward = _backward
        return out

    def __rtruediv__(self, other):
        if type(other) is not Value:
            other = Value(other)
        return other.__truediv__(self)

    def __pow__(self, other):
        out = Value(self.data ** other, {self}, "**")
        def _backward():
            self.grad += out.grad *  other * self.data ** (other - 1)
        out._backward = _backward
        return out

    def tanh(self):
        out = Value(math.tanh(self.data), children={self}, op="tanh")
        def _backward():
            self.grad += out.grad * (1 - math.tanh(self.data) ** 2)
        out._backward = _backward
        return out

In [42]:
a = Value(2)
b = Value(-3)

c = a * b
d = c + a
e = d.tanh()

e.backward()

In [43]:
print(e.grad)
print(d.grad)
print(c.grad)
print(a.grad)
print(b.grad)

1
0.0013409506830258655
0.0013409506830258655
-0.002681901366051731
0.002681901366051731


# Neron

In [45]:
x1 = Value(2)
x2 = Value(3)

w1 = Value(-1)
w2 = Value(2)

b = Value(0.5)

n = x1 * w1 + x2 * w2 + b
o = n.tanh()
o.data

0.9997532108480275

In [46]:
o.backward()
print(x1.grad)
print(x2.grad)
print(w1.grad)
print(w2.grad)
print(b.grad)
print(n.grad)
print(o.grad)

-0.000493517399059451
0.000987034798118902
0.000987034798118902
0.001480552197178353
0.000493517399059451
0.000493517399059451
1


In [ ]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(0.5) for _ in range(nin)]
        self.b = Value(0.5)

# Neouron Forward pass

In [55]:
class Neuron:
    def __init__(self, nin: int):
        self.w = [Value(0.5) for _ in range(nin)]
        self.b = Value(0.5)

    def forward(self, x: list[Value]):
        res = self.b
        for _x, _w in zip(x, self.w):
            res += _x * _w
        return res.tanh()

In [56]:
n = Neuron(3)
x = [Value(2), Value(3), Value(4)]
print(n.forward(x).data)

0.9999092042625951


# Parameters

In [57]:
class Neuron:
    def __init__(self, nin: int):
        self.w = [Value(0.5) for _ in range(nin)]
        self.b = Value(0.5)

    def forward(self, x: list[Value]):
        res = self.b
        for _x, _w in zip(x, self.w):
            res += _x * _w
        return res.tanh()
    
    def parameters(self):
        return self.w + [self.b]

# Layer

In [ ]:
class Layer:
    def __init__(self, nin, nout):
        self.n = [Neuron(nin) for _ in range(nout)]

In [ ]:
Layer(3, 4)

# Forward

In [58]:
class Layer:
    def __init__(self, nin, nout):
        self.n = [Neuron(nin) for _ in range(nout)]

    def forward(self, x):
        res = []
        for _n in self.n:
            res.append(_n.forward(x))
        return res

# Multi layer perceptron

In [ ]:
class MLP:
    def __init__(self, nin, nouts):
        self.l = []
        for nout in nouts:
            self.l.append(Layer(nin, nout))
            nin = nout


In [60]:
mlp = MLP(3, [4, 4, 1])

# Forward pass

In [63]:
class MLP:
    def __init__(self, nin, nouts):
        self.l = []
        for nout in nouts:
            self.l.append(Layer(nin, nout))
            nin = nout

    def forward(self, x):
        for l in self.l:
            x = l.forward(x)
        return x

In [66]:
mlp = MLP(3, [4, 4, 1])
x = [Value(2), Value(3), Value(4)]

out = mlp.forward(x)
print(out)

# Layer parameter

In [68]:
class Layer:
    def __init__(self, nin, nout):
        self.n = [Neuron(nin) for _ in range(nout)]

    def forward(self, x):
        res = []
        for _n in self.n:
            res.append(_n.forward(x))
        return res

    def parameters(self):
        res = []
        for n in self.n:
            res += n.parameters()

        return res

# MLP paramter

In [69]:
class MLP:
    def __init__(self, nin, nouts):
        self.l = []
        for nout in nouts:
            self.l.append(Layer(nin, nout))
            nin = nout

    def forward(self, x):
        for l in self.l:
            x = l.forward(x)
        return x

    def parameters(self):
        res = []
        for l in self.l:
            res += l.parameters()

        return res

In [70]:
mlp = MLP(3, [4, 4, 1])
len(mlp.parameters())

41

# Random initialization of neuron weights and biases

In [71]:
import random

class Neuron:
    def __init__(self, nin: int):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def forward(self, x: list[Value]):
        res = self.b
        for _x, _w in zip(x, self.w):
            res += _x * _w
        return res.tanh()
    
    def parameters(self):
        return self.w + [self.b]

# Neuron, Layer, MLP

In [72]:
import random

class Neuron:
    def __init__(self, nin: int):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def forward(self, x: list[Value]):
        res = self.b
        for _x, _w in zip(x, self.w):
            res += _x * _w
        return res.tanh()
    
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.n = [Neuron(nin) for _ in range(nout)]

    def forward(self, x):
        res = []
        for _n in self.n:
            res.append(_n.forward(x))
        return res

    def parameters(self):
        res = []
        for n in self.n:
            res += n.parameters()

        return res

class MLP:
    def __init__(self, nin, nouts):
        self.l = []
        for nout in nouts:
            self.l.append(Layer(nin, nout))
            nin = nout

    def forward(self, x):
        for l in self.l:
            x = l.forward(x)
        return x

    def parameters(self):
        res = []
        for l in self.l:
            res += l.parameters()

        return res

In [85]:
target = 1
x =[2, 3]
x = [Value(2), Value(3)]
model = MLP(2, [3, 1])
output = model.forward(x)
predition = output[0]
print(output[0].data)

loss = (target - predition) ** 2
print(loss.data)
print()
loss.backward()
parameters = model.parameters()
for parameter in parameters:
    print(parameter.data)
    print(parameter.grad)

0.09993859337218398
0.8101105357008427

-0.7278543810610387
-0.0032026853515745596
-0.729057709658461
-0.0048040280273618395
-0.304634896335521
-0.0016013426757872798
0.10496422280648665
-0.01894153581338234
0.5968931776544637
-0.028412303720073508
0.7303630780254708
-0.00947076790669117
-0.21231770416110995
-1.8013171099502057
0.5071331272212773
-2.7019756649253086
-0.3161694729495774
-0.9006585549751028
0.6033574128773129
1.780816165190695
0.31561193986987646
-1.7670761729586066
0.8811754843730168
-1.1638250159052401
-0.1852119517434696
-1.7821436862295428


In [86]:
print(len(model.parameters()))

13


# Learning Rate

In [88]:
learning_rate = 0.1
target = 1
x =[2, 3]
x = [Value(2), Value(3)]
model = MLP(2, [3, 1])
output = model.forward(x)
predition = output[0]
print(output[0].data)

loss = (target - predition) ** 2
print(loss.data)
print()
loss.backward()
parameters = model.parameters()
for parameter in parameters:
    parameter.data = parameter.data - learning_rate * parameter.grad

0.23499252733344472
0.5852364332356702



# Reset the gradients

In [89]:
for parameter in parameters:
    parameter.grad = 0

# Loop over the training data and train the model

In [94]:

target = 1
x =[2, 3]
x = [Value(2), Value(3)]

model = MLP(2, [3, 1])

learning_rate = 0.1

steps = 10
for step in range(steps):
    output = model.forward(x)
    predition = output[0]
    loss = (target - predition) ** 2
    print(loss.data)
    loss.backward()
    parameters = model.parameters()
    for parameter in parameters:
        parameter.data = parameter.data - learning_rate * parameter.grad
    for parameter in parameters:
        parameter.grad = 0

0.34511234970092414
0.02503835918210203
0.021271095072618394
0.018491774537354764
0.016352500577857046
0.014652954868883803
0.013269145791859166
0.012120005171577594
0.011150195227654439
0.010320601484985795


# Mulitple examples

In [95]:
xs = [
    [Value(2), Value(3)],
    [Value(1), Value(4)],
    [Value(3), Value(2)],
]

ys = [1, 0, 1]


model = MLP(2, [3, 1])

learning_rate = 0.1
steps = 10


for step in range(steps):
    avg = 0
    for x, y in zip(xs, ys):
        output = model.forward(x)
        predition = output[0]
        loss = (y - predition) ** 2
        avg += loss
    avg /= len(xs)
    print(avg.data)
    avg.backward()
    parameters = model.parameters()
    for parameter in parameters:
        parameter.data = parameter.data - learning_rate * parameter.grad
        parameter.grad = 0
        


0.23793181305732522
0.0668154519086856
0.04812986795257557
0.0403073448926631
0.034450724017434556
0.029950637299331073
0.02642594423101075
0.023614808022343184
0.02133483823335383
0.019457053364965712


# Check model actually learned something

In [96]:
xs = [
    [Value(2), Value(3)],
    [Value(1), Value(4)],
    [Value(3), Value(2)],
]

ys = [1, 0, 1]

for x, y in zip(xs, ys):
    p = model.forward(x)
    print((x[0].data, x[1].data), y, p[0].data)

(2, 3) 1 0.7891532518166403
(1, 4) 0 0.059777729548263235
(3, 2) 1 0.9249228497442846


In [98]:
import torch

a = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(2.0, requires_grad=True)

c = a * b
c.backward()

In [99]:
print(a.grad)
print(b.grad)

tensor(2.)
tensor(3.)
